# USD/SOS Exchange Rate Analysis
## Somalia Market Intelligence Platform

This notebook performs comprehensive analysis of USD/SOS exchange rate trends, volatility, and regional variations across Somalia's major cities (2024-2026).

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plot⁸ly.graph_objects as go
import plotly.express as px
from scipy import stats
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

print("✓ Libraries loaded successfully")

SyntaxError: invalid character '⁸' (U+2078) (1656463597.py, line 5)

## 1. Data Loading and Exploration

In [ ]:
# Load exchange rate data
df_rates = pd.read_csv('../data/processed/exchange_rates.csv')
df_rates['date'] = pd.to_datetime(df_rates['date'])

print(f"📊 Exchange Rate Dataset")
print(f"   Records: {len(df_rates):,}")
print(f"   Date range: {df_rates['date'].min().date()} to {df_rates['date'].max().date()}")
print(f"   Cities: {df_rates['city'].nunique()}")
print(f"\n📋 Data sample:")
print(df_rates.head(10))
print(f"\n📈 Exchange rate statistics:")
print(df_rates['usd_sos_rate'].describe())

## 2. Overall Exchange Rate Trends

In [ ]:
# Calculate daily average exchange rate across all cities
daily_avg = df_rates.groupby('date')['usd_sos_rate'].mean().reset_index()

# Create interactive line chart
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_avg['date'],
    y=daily_avg['usd_sos_rate'],
    mode='lines',
    name='Daily Average Rate',
    line=dict(color='#1f77b4', width=2)
))

fig.update_layout(
    title='USD/SOS Exchange Rate Trend (2024-2026)',
    xaxis_title='Date',
    yaxis_title='Exchange Rate (SOS per USD)',
    hovermode='x unified',
    height=400,
    template='plotly_white'
)

fig.write_html('../visuals/exchange_rate_trend.html')
fig.show()

print(f"✓ Exchange rate trend chart saved")

## 3. Moving Averages and Trend Analysis

In [ ]:
# Calculate moving averages
daily_avg['ma7'] = daily_avg['usd_sos_rate'].rolling(window=7, min_periods=1).mean()
daily_avg['ma30'] = daily_avg['usd_sos_rate'].rolling(window=30, min_periods=1).mean()
daily_avg['ma90'] = daily_avg['usd_sos_rate'].rolling(window=90, min_periods=1).mean()

# Visualization
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_avg['date'],
    y=daily_avg['usd_sos_rate'],
    mode='lines',
    name='Daily Rate',
    line=dict(color='rgba(31, 119, 180, 0.3)', width=1)
))

fig.add_trace(go.Scatter(
    x=daily_avg['date'],
    y=daily_avg['ma7'],
    mode='lines',
    name='7-Day MA',
    line=dict(color='orange', width=2)
))

fig.add_trace(go.Scatter(
    x=daily_avg['date'],
    y=daily_avg['ma30'],
    mode='lines',
    name='30-Day MA',
    line=dict(color='red', width=2)
))

fig.add_trace(go.Scatter(
    x=daily_avg['date'],
    y=daily_avg['ma90'],
    mode='lines',
    name='90-Day MA',
    line=dict(color='green', width=2)
))

fig.update_layout(
    title='Exchange Rate with Moving Averages',
    xaxis_title='Date',
    yaxis_title='Exchange Rate (SOS per USD)',
    hovermode='x unified',
    height=500,
    template='plotly_white'
)

fig.write_html('../visuals/exchange_rate_moving_averages.html')
fig.show()

print(f"✓ Moving averages chart saved")

## 4. Volatility Analysis

In [ ]:
# Calculate rolling volatility (30-day rolling standard deviation)
daily_avg['volatility_30d'] = daily_avg['usd_sos_rate'].rolling(window=30, min_periods=1).std()

# Visualization
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=daily_avg['date'],
    y=daily_avg['volatility_30d'],
    mode='lines',
    name='30-Day Rolling Volatility',
    line=dict(color='#d62728', width=2),
    fill='tozeroy'
))

fig.update_layout(
    title='Exchange Rate Volatility (30-Day Rolling Std Dev)',
    xaxis_title='Date',
    yaxis_title='Standard Deviation',
    hovermode='x unified',
    height=400,
    template='plotly_white'
)

fig.write_html('../visuals/exchange_rate_volatility.html')
fig.show()

print(f"Volatility Statistics:")
print(f"   Mean volatility: {daily_avg['volatility_30d'].mean():.4f}")
print(f"   Max volatility: {daily_avg['volatility_30d'].max():.4f}")
print(f"   Min volatility: {daily_avg['volatility_30d'].min():.4f}")

## 5. Regional Comparison

In [ ]:
# Recent rates by city (last 30 days)
recent_cutoff = df_rates['date'].max() - timedelta(days=30)
recent_rates = df_rates[df_rates['date'] >= recent_cutoff]

city_avg = recent_rates.groupby('city')['usd_sos_rate'].agg(['mean', 'std', 'min', 'max']).reset_index()
city_avg = city_avg.sort_values('mean', ascending=False)

print(f"📍 Exchange Rates by City (Last 30 Days):")
print(city_avg.to_string(index=False))

# Visualization
fig = px.bar(
    city_avg,
    x='city',
    y='mean',
    error_y='std',
    color='mean',
    color_continuous_scale='Viridis',
    title='Average Exchange Rate by City (Last 30 Days)',
    labels={'mean': 'Exchange Rate (SOS/USD)', 'city': 'City'},
    height=400
)

fig.update_layout(template='plotly_white')
fig.write_html('../visuals/exchange_rate_by_city.html')
fig.show()

print(f"✓ Regional comparison chart saved")

## 6. Market Type Analysis (Black Market vs Official)

In [ ]:
# Compare market types
market_comp = df_rates.groupby(['date', 'market_type'])['usd_sos_rate'].mean().reset_index()

fig = go.Figure()

for market_type in market_comp['market_type'].unique():
    data = market_comp[market_comp['market_type'] == market_type]
    fig.add_trace(go.Scatter(
        x=data['date'],
        y=data['usd_sos_rate'],
        mode='lines',
        name=market_type,
        line=dict(width=2)
    ))

fig.update_layout(
    title='Exchange Rate Comparison: Official vs Black Market',
    xaxis_title='Date',
    yaxis_title='Exchange Rate (SOS per USD)',
    hovermode='x unified',
    height=400,
    template='plotly_white'
)

fig.write_html('../visuals/market_type_comparison.html')
fig.show()

print("✓ Market type comparison chart saved")

## 7. Key Insights and KPIs

In [ ]:
# Calculate KPIs
print("\n" + "="*70)
print("KEY PERFORMANCE INDICATORS - USD/SOS EXCHANGE RATE")
print("="*70)

# Overall statistics
current_rate = daily_avg['usd_sos_rate'].iloc[-1]
start_rate = daily_avg['usd_sos_rate'].iloc[0]
rate_change = current_rate - start_rate
rate_change_pct = (rate_change / start_rate) * 100

print(f"\n💱 Exchange Rate Dynamics:")
print(f"   Current rate: {current_rate:.2f} SOS/USD")
print(f"   Starting rate (Jan 2024): {start_rate:.2f} SOS/USD")
print(f"   Total change: {rate_change:.2f} SOS ({rate_change_pct:.2f}%)")
print(f"   Average daily change: {df_rates['daily_change_percent'].mean():.4f}%")

# Volatility
print(f"\n📊 Volatility Metrics:")
print(f"   Mean volatility (30-day): {daily_avg['volatility_30d'].mean():.4f}")
print(f"   Peak volatility date: {daily_avg.loc[daily_avg['volatility_30d'].idxmax(), 'date'].date()}")
print(f"   Peak volatility value: {daily_avg['volatility_30d'].max():.4f}")

# Regional insights
print(f"\n📍 Regional Insights:")
most_stable = city_avg.loc[city_avg['std'].idxmin()]
least_stable = city_avg.loc[city_avg['std'].idxmax()]
print(f"   Most stable city: {most_stable['city']} (σ = {most_stable['std']:.4f})")
print(f"   Least stable city: {least_stable['city']} (σ = {least_stable['std']:.4f})")
print(f"   Highest rate: {city_avg.iloc[0]['city']} ({city_avg.iloc[0]['mean']:.2f} SOS/USD)")
print(f"   Lowest rate: {city_avg.iloc[-1]['city']} ({city_avg.iloc[-1]['mean']:.2f} SOS/USD)")

print("\n" + "="*70)

## 8. Recommendations and Insights

In [ ]:
insights = f"""
📈 EXCHANGE RATE ANALYSIS - STRATEGIC INSIGHTS

1. CURRENCY DEPRECIATION TREND
   • SOS has depreciated {rate_change_pct:.2f}% against USD over the analysis period
   • This reflects broader currency pressures in the Somali economy
   • Particularly affects import-dependent sectors and increases costs for essential goods

2. VOLATILITY PATTERNS
   • Exchange rate volatility averages {daily_avg['volatility_30d'].mean():.4f}, indicating moderate market instability
   • Peak volatility suggests periods of heightened economic uncertainty
   • This creates hedging demands from businesses and investors

3. MARKET TYPE DIFFERENTIALS
   • Official and black market rates show significant divergence
   • Suggests capital controls or official rate management
   • Arbitrage opportunities exist between market segments

4. REGIONAL VARIATIONS
   • Most stable: {most_stable['city']} (σ={most_stable['std']:.4f})
   • Least stable: {least_stable['city']} (σ={least_stable['std']:.4f})
   • Indicates different market depth and economic conditions across regions

5. BUSINESS IMPLICATIONS
   • Import companies should consider forward contracts to hedge currency risk
   • Remittance flows are highly sensitive to exchange rate movements
   • Tourism and international services revenue face headwinds from weak SOS

6. POLICY RECOMMENDATIONS
   • Monitor exchange rate pressure as leading indicator of inflation
   • Consider interventions to reduce unnecessary volatility
   • Strengthen market transparency and integration
"""

print(insights)

# Save insights
with open('../reports/exchange_rate_insights.txt', 'w') as f:
    f.write(insights)

print("✓ Insights saved to reports/")